In [ ]:
import time
import random
import csv
import os
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait

BASE_URL = "https://www.indiabix.com/data-interpretation/questions-and-answers/"
CSV_FILE = "Indiabix_all_charts.csv"

MIN_DELAY = 2
MAX_DELAY = 4

def human_delay():
    time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))

def small_delay():
    time.sleep(random.uniform(1, 2))
#Creating driver to open chrome automatically.
def create_driver():
    options = Options()
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)

    driver = webdriver.Chrome(options=options)
    driver.execute_script(
        "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
    )
    return driver

#Creating csv file to store data
def setup_csv():
    if not os.path.exists(CSV_FILE):
        with open(CSV_FILE, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow([
                "chart_type",
                "exercise",
                "page_number",
                "question",
                "option_a",
                "option_b",
                "option_c",
                "option_d",
                "correct_answer",
            ])

def save_to_csv(data):
    with open(CSV_FILE, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(data)
#To handle pages in the website.
def go_to_next_page(driver):
    try:
        next_li = driver.find_element(
            By.XPATH,
            "//ul[contains(@class,'pagination')]//li[.//a[normalize-space()='Next']]"
        )

        if "disabled" in next_li.get_attribute("class").lower():
            return False

        next_link = next_li.find_element(By.TAG_NAME, "a")
        driver.execute_script("arguments[0].click();", next_link)
        time.sleep(3)
        return True

    except:
        return False
#Scraping the data -Data Interpretation
def scrape_exercise(driver, chart_type, exercise_name):

    wait = WebDriverWait(driver, 15)
    page_number = 1

    print(f"\n{chart_type} | {exercise_name} STARTED")

    while True:

        human_delay()

        question_blocks = wait.until(
            lambda d: d.find_elements(By.CLASS_NAME, "bix-div-container")
        )

        print(f"{chart_type} | {exercise_name} | Page {page_number} → {len(question_blocks)} questions")
        #Scraping questions
        for block in question_blocks:

            try:
                question_text = block.find_element(
                    By.CLASS_NAME, "bix-td-qtxt"
                ).text.strip()
            except:
                continue
         #Scraping options
            option_rows = block.find_elements(By.CLASS_NAME, "bix-opt-row")

            option_texts = []
            correct_answer = ""
            
            # Collect options first
            for row in option_rows:
                try:
                    option_val = row.find_element(By.CLASS_NAME, "bix-td-option-val")
                    option_texts.append(option_val.text.strip())
                except:
                    option_texts.append("")

            while len(option_texts) < 4:
                option_texts.append("")

            # Click ALL options
            for row in option_rows:
                try:
                    option_btn = row.find_element(By.CLASS_NAME, "bix-td-option")
                    driver.execute_script("arguments[0].click();", option_btn)
                    small_delay()
                except:
                    continue

            # Wait for UI update
            time.sleep(1)

            # Detect correct option via class change
            for row in option_rows:
                try:
                    option_val = row.find_element(By.CLASS_NAME, "bix-td-option-val")
                    classes = option_val.get_attribute("class") or ""

                    if "correct" in classes.lower() or "right" in classes.lower():
                        correct_answer = option_val.text.strip()
                        break
                except:
                    continue
            save_to_csv([
                chart_type,
                exercise_name,
                page_number,
                question_text,
                option_texts[0],
                option_texts[1],
                option_texts[2],
                option_texts[3],
                correct_answer,
            ])

            # if correct_answer:
            #     print(f"  ✓ {question_text[:40]}... → {correct_answer}")
            # else:
            #     print(f"  ? {question_text[:40]}... → No answer detected")

        moved = go_to_next_page(driver)

        if not moved:
            break

        page_number += 1

    print(f" {chart_type} | {exercise_name} COMPLETED")

def main():

    setup_csv()
    driver = create_driver()
    wait = WebDriverWait(driver, 15)

    driver.get(BASE_URL)
    human_delay()

    # Find ALL chart types
    chart_links = wait.until(
        lambda d: d.find_elements(By.XPATH, "//ul[@class='need-ul-filter']//a")
    )

    chart_types = {}
    for c in chart_links:
        chart_name = c.text.strip()
        if chart_name and any(keyword in chart_name.lower() for keyword in 
            ['table', 'line', 'bar', 'pie', 'radar', 'mixed']):
            chart_types[chart_name] = c.get_attribute("href")
            print(f"Found chart type: {chart_name}")

    print(f"\n  Starting ALL CHARTS scraping ({len(chart_types)} types)...")

    for chart_type, chart_url in chart_types.items():

        print(f"\n{'='*70}")
        print(f" PROCESSING CHART TYPE: {chart_type}")
        print(f"{'='*70}")

        driver.get(chart_url)
        human_delay()

        exercise_links = driver.find_elements(
            By.XPATH,
            "//a[contains(@href,'/data-interpretation/') and string-length(normalize-space(text())) > 0]"
        )

        exercise_data = []

        for e in exercise_links:
            name = e.text.strip()
            url = e.get_attribute("href")

            if name and name[-1].isdigit():
                exercise_data.append((name, url))

        print(f" Found {len(exercise_data)} exercises for {chart_type}")

        for exercise_name, exercise_url in exercise_data:

            print(f"\n📖 Exercise: {exercise_name}")
            driver.get(exercise_url)
            human_delay()

            scrape_exercise(driver, chart_type, exercise_name)

        print(f"\n {chart_type} COMPLETED")

    driver.quit()

    print("\n ALL CHART TYPES SCRAPING COMPLETED!")
    print(" Saved at:", os.path.abspath(CSV_FILE))


if __name__ == "__main__":
    main()
